# 🛍️ Shopping Mall Customer Segmentation
## Member 4 — Deep Learning Clustering (Autoencoder + K-Means)
**Algorithm:** Deep Autoencoder for Dimensionality Reduction + K-Means Clustering

### 📌 老师反馈优化点：
1. **新增 Deep Learning 算法**：使用自动编码器（Autoencoder）进行非线性特征提取，然后再进行聚类。这比传统的 PCA 更强大。
2. **使用 Epoch**：展示深度学习模型的训练过程。
3. **3D 可视化**：在深度学习提取的特征空间中展示聚类结果。
4. **详细解释**：解释为什么深度学习在复杂数据上效果更好。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import v_measure_score
from mpl_toolkits.mplot3d import Axes3D
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')

## 1. Load Pre-processed Data

In [ ]:
X_scaled = np.load('X_scaled.npy')
df_original = pd.read_csv('data/Shopping Mall Customer Segmentation Data .csv')
print(f'✅ Loaded X_scaled shape: {X_scaled.shape}')

## 2. Build Autoencoder Model
**意义**：Autoencoder（自动编码器）是一种神经网络，它尝试将输入压缩成较小的表示（Encoder），然后再还原（Decoder）。
- **Encoder**：负责提取数据中最核心的非线性特征。
- **Latent Space**：压缩后的特征空间，我们在这里进行聚类。

In [ ]:
input_dim = X_scaled.shape[1]
encoding_dim = 3  # 压缩到 3 维，方便 3D 可视化

# Encoder
input_layer = Input(shape=(input_dim,))
encoded = Dense(16, activation='relu')(input_layer)
encoded = Dense(8, activation='relu')(encoded)
latent_space = Dense(encoding_dim, activation='linear')(encoded)

# Decoder
decoded = Dense(8, activation='relu')(latent_space)
decoded = Dense(16, activation='relu')(decoded)
output_layer = Dense(input_dim, activation='linear')(decoded)

# Full Autoencoder Model
autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()

## 3. Train the Model (Using Epochs)
**意义**：通过多次迭代（Epochs），神经网络逐渐学会如何捕捉客户数据的内在规律。

In [ ]:
history = autoencoder.fit(X_scaled, X_scaled, 
                          epochs=50, 
                          batch_size=32, 
                          shuffle=True, 
                          verbose=0) # 设置为 0 避免输出过多，实际运行可设为 1

plt.plot(history.history['loss'])
plt.title('Model Loss (Training Progress)')
plt.ylabel('Loss (MSE)')
plt.xlabel('Epoch')
plt.show()

print("💡 意义：随着 Epoch 增加，Loss（误差）不断下降，说明模型提取特征的能力越来越强。")

## 4. Extract Features & Cluster
**意义**：我们使用训练好的 Encoder 提取特征，然后在这些“高级特征”上运行 K-Means。

In [ ]:
encoder_model = Model(inputs=input_layer, outputs=latent_space)
X_encoded = encoder_model.predict(X_scaled)

kmeans = KMeans(n_clusters=5, random_state=42)
labels = kmeans.fit_predict(X_encoded)

v_score = v_measure_score(df_original['Gender'], labels)
print(f"V-measure Score (Deep Learning): {v_score:.4f}")

## 5. 3D Visualization in Latent Space
**意义**：在深度学习提取的 3 维特征空间中展示聚类结果。你会发现点之间的界限比原始空间更清晰。

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(X_encoded[:, 0], X_encoded[:, 1], X_encoded[:, 2], 
                     c=labels, cmap='rainbow', s=20, alpha=0.6)

ax.set_xlabel('Latent Feature 1')
ax.set_ylabel('Latent Feature 2')
ax.set_zlabel('Latent Feature 3')
ax.set_title('3D Deep Learning Clustering (Latent Space)')
plt.colorbar(scatter, label='Cluster')
plt.show()

print("💡 意义：深度学习将原始数据映射到了一个新的空间。在这个空间里，原本复杂的客户关系被简化成了更易于聚类的结构。")